In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, count

spark = SparkSession.builder.getOrCreate()

standard_sites = spark.read.table(f"{SILVER}.site_standardized")
unmapped = standard_sites.filter(col("is_standardized") == False) \
    .select(col("raw_city").alias("City"), col("raw_country").alias("Country")) \
    .groupBy("City", "Country").agg(count("*").alias("Affected_Sites")) \
    .withColumn("Export_Date", current_timestamp())

unmapped.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.ba_unmapped_tracker")

export_path = f"{LANDING}/ba_feedback/unmapped_latest.csv"
unmapped.coalesce(1).write.mode("overwrite").option("header", "true").csv(export_path)

print("✅ BA Feedback export done!")